# CS 6220 - Homework 2
## Zihe Hao

### Part 1: Data Generator

In [ ]:
import math
import numpy as np

def generate_noisy_dataset(functions, true_function, x_range, n_samples, epsilon, seed=67):
    low, high = map(float, x_range)

    rng = np.random.default_rng(seed)

    x = rng.uniform(low, high, n_samples)
    X = make_features(x, functions)

    truth = np.array([true_function(float(value)) for value in x], dtype=float)
    y = truth + rng.uniform(-epsilon, epsilon, n_samples)
    
    return np.column_stack((X, y))


def make_features(x, functions=()):
    x = np.asarray(x, dtype=float).reshape(-1)
    columns = [x]
    for function in functions:
        values = np.array([function(float(value)) for value in x], dtype=float)
        columns.append(values)
    X = np.column_stack(columns)
    return X

demo = generate_noisy_dataset(
    [lambda x: x ** 2, lambda x: x ** 3], math.log,
    (0.01, 1.0), 10, 0.1, seed=7
)
print('Example columns: x, x^2, x^3, y')
print(demo[:5])


Example columns: x, x^2, x^3, y
[[ 0.62884451  0.39544542  0.24867368 -0.50326477]
 [ 0.89824166  0.80683809  0.72473558 -0.15163101]
 [ 0.77792883  0.60517327  0.47078174 -0.30014632]
 [ 0.23295512  0.05426809  0.01264203 -1.46789421]
 [ 0.30716462  0.09435011  0.02898101 -1.1794618 ]]


### Part 2: Linear regression


In [5]:
def fit(dataset):
    data = checked_dataset(dataset)
    X_base, y = data[:, :-1], data[:, -1]
    X = np.column_stack((np.ones(len(X_base)), X_base))

    left_hand_side = X.T @ X
    right_hand_side = X.T @ y
    return np.linalg.solve(left_hand_side, right_hand_side)


def predict(weights, test_data):
    weights = np.asarray(weights, dtype=float).reshape(-1)
    X = np.asarray(test_data, dtype=float)
    n_features = len(weights) - 1
    if X.ndim == 1 and n_features == 1:
        X = X.reshape(-1, 1)

    if X.shape[1] == n_features + 1:
        X = X[:, :-1]  # Last column is y, not part of features

    X__with_constants = np.column_stack((np.ones(len(X)), X))
    return X_with_constants @ weights


def evaluate_model(predictions, true_outputs):
    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    true_outputs = np.asarray(true_outputs, dtype=float).reshape(-1)
    return float(np.sqrt(np.mean((predictions - true_outputs) ** 2)))


### Part 3: Regression Tree

In [7]:
from dataclasses import dataclass
import matplotlib.pyplot as plt

@dataclass
class TreeNode:
    prediction: float
    n_records: int
    feature: int = -1
    threshold: float = 0
    left: object = None
    right: object = None

    @property
    def is_leaf(self):
        return self.left is None


class RegressionTree:
    def __init__(self, feature_functions=()):
        self.feature_functions = list(feature_functions)
        self.root = None

    def fit(self, dataset, min_records):
        data = np.asarray(dataset, dtype=float)
        self.min_records = min_records
        self.n_features = data.shape[1] - 1
        self.training_data = data
        self.root = self._build(data[:, :-1], data[:, -1])
        return self

    def _best_split(self, X, y):
        best_loss = np.var(y)
        best = None

        for feature in range(X.shape[1]):
            values = np.unique(X[:, feature])
            thresholds = (values[:-1] + values[1:]) / 2

            for threshold in thresholds:
                mask = X[:, feature] <= threshold
                left, right = y[mask], y[~mask]
                loss = (len(left) * np.var(left) +
                        len(right) * np.var(right)) / len(y)

                if loss < best_loss - 1e-12:
                    best_loss = loss
                    best = (feature, threshold)

        return best

    def _build(self, X, y):
        node = TreeNode(float(np.mean(y)), len(y))

        if len(y) < max(2, self.min_records):
            return node

        split = self._best_split(X, y)
        if split is not None:
            node.feature, node.threshold = split
            mask = X[:, node.feature] <= node.threshold
            node.left = self._build(X[mask], y[mask])
            node.right = self._build(X[~mask], y[~mask])

        return node

    def predict(self, X):
        X = np.asarray(X, dtype=float).reshape(-1, self.n_features)
        predictions = []

        for row in X:
            node = self.root
            while not node.is_leaf:
                node = node.left if row[node.feature] <= node.threshold else node.right
            predictions.append(node.prediction)

        return np.array(predictions)

    def statistics(self):
        def walk(node, depth=0):
            if node.is_leaf:
                return 1, 1, depth, [node.n_records]

            a = walk(node.left, depth + 1)
            b = walk(node.right, depth + 1)
            return (1 + a[0] + b[0], a[1] + b[1],
                    max(a[2], b[2]), a[3] + b[3])

        nodes, leaves, depth, counts = walk(self.root)
        return dict(nodes=nodes, leaves=leaves, depth=depth,
                    leaf_records=counts)

    def visualize(self, true_function=None, x_range=(0, 1),
                  title='Regression tree', subtitle=''):
        x = np.linspace(*x_range, 1000)
        X = np.column_stack([x] + [f(x) for f in self.feature_functions])

        plt.figure(figsize=(9, 5))
        plt.scatter(self.training_data[:, 0], self.training_data[:, -1],
                    s=15, alpha=0.4, label='Training data')
        plt.plot(x, self.predict(X), label='Tree prediction')

        if true_function is not None:
            plt.plot(x, [true_function(v) for v in x],
                     label='True function')

        plt.title(title + ('\n' + subtitle if subtitle else ''))
        plt.legend()
        plt.show()